# JMI — Notebook 01: Data Ingestion & Data Cleaning Pipeline

**Purpose:**
This notebook demonstrates the data engineering pipeline designed to ingest raw job market postings, clean messy text strings, handle null values, normalize multi-format salaries (Lakhs Per Annum vs Annual vs Hourly), and validate data quality metrics (Completeness, Consistency, Uniqueness, Validity).


In [ ]:
import pandas as pd
import numpy as np
import re
import os

# Load raw dataset
raw_data_path = '../data/raw/sample_jobs.csv'
if os.path.exists(raw_data_path):
    df_raw = pd.read_csv(raw_data_path)
    print(f'Loaded {len(df_raw)} raw job records.')
    display(df_raw.head(3))
else:
    print('Run backend/export_data.py to generate sample CSVs.')


## 1. Salary Parsing & Normalization to LPA
Salary postings come in varied formats: `8 - 14 LPA`, `₹ 12,00,000 / year`, `15 Lakhs`. We implement regex tokenization to convert all numeric forms into standardized Lakhs Per Annum (INR).


In [ ]:
def parse_salary_lpa(sal_str):
    if pd.isna(sal_str):
        return None, None
    sal_str = str(sal_str).strip()
    match = re.search(r'₹?\s*(\d+(?:\.\d+)?)\s*(?:-|to)\s*₹?\s*(\d+(?:\.\d+)?)\s*(?:lpa|lakh)', sal_str, re.IGNORECASE)
    if match:
        return float(match.group(1)), float(match.group(2))
    return None, None

print('Test parser:', parse_salary_lpa('₹10.5 - ₹18.0 LPA'))


## 2. Role Normalization Engine
Unstructured job titles like `Lead BI Specialist` or `Business Data Analyst` are mapped to canonical role categories: `Data Analyst`, `Data Scientist`, `Data Engineer`, `ML Engineer`, `BI Developer`, `Software Engineer`.


In [ ]:
def normalize_role(title):
    t = str(title).lower()
    if 'analyst' in t:
        return 'Data Analyst'
    elif 'scientist' in t:
        return 'Data Scientist'
    elif 'data engineer' in t or 'pipeline' in t:
        return 'Data Engineer'
    elif 'ml' in t or 'learning' in t or 'ai' in t:
        return 'ML Engineer'
    elif 'bi' in t or 'tableau' in t or 'power bi' in t:
        return 'BI Developer'
    return 'Software Engineer'

print('Normalized:', normalize_role('Sr. Business Data Analyst'))


## 3. Data Quality Metrics KPI Computation
We evaluate Completeness, Uniqueness, and Consistency across all columns.


In [ ]:
completeness = (1 - df_raw.isnull().sum().sum() / df_raw.size) * 100
print(f'Dataset Completeness Score: {completeness:.2f}%')
